<a href="https://colab.research.google.com/github/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_2/lessons/lesson_20_classes_namespace/note_lesson_20_classes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Урок 20 — Класи, простір імен: сервіс доставки тримає свої правила

У фінальному проєкті модуля 1 дані (`orders`) жили окремо від функцій, і правила — «номери видає лише сервіс», «одна доставка на замовлення» — можна було оминути, просто дописавши в список. Сьогодні стан і правила переїжджають в один **клас**.

| Розділ | Що всередині | Приклади | Задача |
|---|---|---|---|
| 1 | усе — об'єкти: `type`, `isinstance`, MRO, `type` і `object` | 1.1–1.6 | |
| 2 | картка класу → `class Order`: `__init__`, `self`, методи, `__repr__` | 2.1–2.8 | **Задача 1** |
| 3 | атрибути класу, простір імен `__dict__`, пастки зі спільними даними | 3.1–3.8 | |
| 4 | правила в `__init__`, `@classmethod`, `@staticmethod` | 4.1–4.5 | |
| 5 | `isinstance` і `type` | 5.1–5.2 | |
| 6 | архітектура: три способи тримати стан | 6.1–6.2 | |
| 7 | розібраний приклад: `DeliveryService` і шари системи | 7.1–7.6 | **Задача 2** |
| 8 | промокод: від замикання до класу | 8.1 | **Задача 3** |

**Як працювати**

- Виконуй клітинки **зверху вниз**. Клітинка з визначенням (`class`, `def`) лише створює клас чи функцію — **запуск завжди в наступній клітинці**. Так приклад можна перезапускати, не перевизначаючи клас.
- Перед **🔮 Прогнозом** спершу відповідай сам, потім розгортай відповідь.
- Задач лише три; решта — розібрані приклади: прочитай, запусти, зміни щось і запусти знову.
- Теорія, схеми архітектури й порівняння рішень — у книзі: [Урок 20. Класи, простір імен](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m2/lesson_20/).

## 🔁 Пригадай (без підглядання)

1. Що надрукує `b = a; b.append(4); print(a)` для `a = [1, 2, 3]`?
2. Що зберігає замикання `next_id` з `make_counter()` уроку 19?
3. Чим `NamedTuple` відрізняється від словника?

<details>
<summary>Відповіді</summary>

1. `[1, 2, 3, 4]` — два ярлики одного об'єкта.
2. Змінну `count` оточуючої функції.
3. Поля фіксовані, доступ через крапку, кортеж незмінний.

</details>

---
## 1. Усе — об'єкти

Число, рядок, список, функція — усе це **об'єкти**. У кожного об'єкта є **тип**, або **клас**: він визначає, що з об'єктом можна робити. Клас — креслення, а об'єкти (**екземпляри**) — конкретні речі, зроблені за ним.

### Приклад 1.1. У кожного значення є клас

In [ ]:
print(type(540), type("Поділ"), type([540, 320]))
print(isinstance(540, object), isinstance(len, object))

`type(x)` повертає клас значення. `isinstance(x, object)` — `True` і для числа, і для вбудованої функції `len`: усе в Python — об'єкт. `"Поділ"` і `"Оболонь"` — два екземпляри одного класу `str`.

**🔮 Прогноз:** а який тип у самого `int`?

<details>
<summary>Відповідь</summary>

`<class 'type'>`: клас — теж об'єкт, і його клас — `type`.

</details>

### Приклад 1.2. Клас — теж об'єкт

In [ ]:
print(type(int), type(str))
print(type(int) is type(str))

`int` і `str` — об'єкти класу `type`. Класи можна класти в змінні, списки й словники, як будь-які інші значення (у розділі 5 уроку 21 так зробимо таблицю класів доставки).

### Приклад 1.3. MRO: де клас шукає свої методи

**MRO** (method resolution order) — список класів, у яких Python шукає метод, по порядку. Подивитися його можна трьома способами — результат однаковий:

In [ ]:
print("1. метод mro():       ", int.mro(), str.mro())
print("2. атрибут __mro__:   ", int.__mro__, str.__mro__)
print("3. через метаклас type:", type.mro(int), type.mro(str))

- `int.mro()` повертає **список**, `int.__mro__` — **кортеж** з тими самими класами;
- `type.mro(int)` — той самий метод, викликаний явно через клас `type` (як `str.upper("поділ")` замість `"поділ".upper()`, приклад 1.6);
- у кінці кожного ланцюжка — `object`: спільний предок усіх класів.

### Приклад 1.4. Свої класи теж мають MRO

`class Dog(Animal)` — клас `Dog`, батьком якого є `Animal`. Наслідування докладно — в уроці 21; тут лише дивимось, як виглядає ланцюжок.

In [ ]:
class Animal:
    pass


class Dog(Animal):
    pass

In [ ]:
print(Dog.mro())
print(Dog.__bases__)
print(isinstance(Dog(), Animal), isinstance(Dog(), object))

- `Dog.mro()` — `Dog → Animal → object`: навіть порожній клас без явного батька наслідує `object`;
- `__bases__` — лише **безпосередні** батьки (кортеж з одного `Animal`);
- екземпляр `Dog()` — водночас і `Animal`, і `object`.

### Приклад 1.5. `type` і `object`: хто чий

Два особливі класи стоять в основі всього. **🔮 Прогноз:** який клас у `type`? І чи є `object` екземпляром `type`?

<details>
<summary>Відповідь</summary>

`type(type)` — це сам `type`: він єдиний клас, що є своїм власним класом. І так, `object` — екземпляр `type` (як і будь-який клас), а `type` — екземпляр `object` (як і будь-який об'єкт).

</details>

In [ ]:
print("type(type)       :", type(type), type.__class__ is type)
print("type.__bases__   :", type.__bases__)
print("type.__mro__     :", type.__mro__)
print("object.__class__ :", object.__class__)
print("object.__bases__ :", object.__bases__)
print("object.__mro__   :", object.__mro__)
print("type — об'єкт?   ", isinstance(type, object))
print("object — клас?   ", isinstance(object, type))

Дві різні «осі»:

```text
вісь «чий екземпляр» (__class__)       вісь «хто батько» (__bases__)
540     →  int                         int    →  object
int     →  type                        type   →  object
object  →  type                        object →  ()   (батьків немає: корінь)
type    →  type  (сам собі клас)
```

- `object` — **корінь наслідування**: у нього порожні `__bases__`, а MRO складається з нього самого;
- `type` — **клас усіх класів** (метаклас): `type(int)`, `type(object)` і навіть `type(type)` — це `type`.

### Приклад 1.6. Метод живе в класі

Чому `"поділ".upper()` працює? Бо в класу `str` є метод `upper`. Виклик через крапку на екземплярі — скорочений запис виклику методу класу:

In [ ]:
district = "поділ"
print(district.upper(), str.upper(district))

prices = [540]
list.append(prices, 320)          # те саме, що prices.append(320)
print(prices)
print("upper" in str.__dict__, "append" in list.__dict__)

`district.upper()` і `str.upper(district)` — одне й те саме: метод лежить у класі, а об'єкт передається йому **першим аргументом**. Запам'ятай цей рядок — у розділі 2 він пояснить, що таке `self`.

Досі ми користувалися чужими класами. Тепер спроєктуємо й напишемо свій.

---
## 2. Спершу проєктуємо, потім пишемо: клас `Order`

Готові класи хтось колись спроєктував. `list` **знає** елементи, **вміє** `append` і `sort` і **змінюється** сам. `tuple` знає елементи, але `append` у нього немає, бо його **правило** — незмінність. Свій клас починаємо так само: не з `class`, а з **картки класу** — п'яти запитань.

| Запитання | Замовлення кафе | Що з'явиться в коді |
|---|---|---|
| **1. Що це за річ?** | замовлення; кожне окреме | `class Order:`; кожне замовлення — екземпляр `Order(...)` |
| **2. Що воно знає?** | номер, час, чек, чайові, гостей | атрибути в `__init__`: `self.bill = bill` |
| **3. Що воно вміє?** | сума з чайовими, сума на гостя, показати себе | методи `total()`, `per_guest()`, `__repr__()` |
| **4. Які правила тримає?** | чек > 0, гостей ≥ 1 | перевірки в `__init__` (розділ 4) |
| **5. Чого НЕ робить?** | не видає собі номер, не знає про доставку і `print` | номер видає сервіс (розділ 7) |

```text
предмет        →  картка класу   →  class Order   →  order = Order(...)  →  order.total()
(замовлення       (проєктування:     (креслення)      (екземпляр:            (метод працює з даними
 за столиком)      коду ще немає)                      свої атрибути)         саме цього order)
```

Іменник → **клас**, «знає» → **атрибути**, «вміє» → **методи**, правила → **перевірки**, «не робить» → межа відповідальності.

Три типові помилки проєктування:

- **почати з коду** — методи з'являються «бо треба», і клас виростає в смітник функцій;
- **клас без жодного «вміє»** — якщо об'єкт лише зберігає дані, `NamedTuple` з уроку 5 може бути простішим;
- **клас, що робить усе** — читає файл, рахує, друкує й надсилає SMS. Відповідь на п'яте запитання має бути непорожньою.

### Приклад 2.1. Картка → код

«Знає» стає рядками `__init__`, «вміє» — методами. Правила додамо в розділі 4.

In [ ]:
from datetime import datetime


class Order:
    """Замовлення кафе."""

    def __init__(self, order_id, time, bill, tip, guests):
        self.id = order_id
        self.time = time
        self.bill = bill
        self.tip = tip
        self.guests = guests

    def total(self):
        """Чек разом з чайовими."""
        return self.bill + self.tip

    def per_guest(self):
        return round(self.bill / self.guests, 2)

In [ ]:
order = Order(1, datetime(2024, 7, 19, 18, 30), 540.0, 50.0, 2)
print(type(order), order.__class__.mro())
print(order.bill, order.total(), order.per_guest())

- **`class Order:`** оголошує клас — новий тип, як `int` чи `list`; його MRO — `Order → object`;
- **`Order(1, …)`** створює екземпляр: Python робить порожній об'єкт і викликає `__init__`, щоб його заповнити;
- **`self.bill = bill`** кладе значення в **цей** об'єкт — це атрибут екземпляра;
- **`order.total()`** — виклик методу конкретного замовлення: 540 + 50 = 590.0; `per_guest()` — 540 / 2 = 270.0.

### Приклад 2.2. Два екземпляри — два набори даних

**🔮 Прогноз:** створимо друге замовлення з **тими самими** даними. Чи буде `order_2 == order`? А `order_2 is order`?

<details>
<summary>Відповідь</summary>

Обидва `False`. Кожен виклик `Order(...)` створює **новий** об'єкт. Без спеціального методу `__eq__` (урок 24) `==` для своїх класів порівнює, чи це той самий об'єкт, — тобто працює як `is`.

</details>

In [ ]:
order_2 = Order(1, datetime(2024, 7, 19, 18, 30), 540.0, 50.0, 2)
lunch = Order(2, datetime(2024, 7, 19, 12, 10), 320.0, 30.0, 1)

print(order_2 == order, order_2 is order)
print(order.total(), order_2.total(), lunch.total())
print(lunch.__dict__)

Атрибути кожного екземпляра лежать у його власному словнику `__dict__`. Методи (`total`, `per_guest`) там немає — вони одні на всіх і живуть у класі.

```text
class Order: __init__, total, per_guest      ← методи: один раз, у класі
   ├── order   {'id': 1, 'bill': 540.0, ...}  ← дані: у кожного свої
   ├── order_2 {'id': 1, 'bill': 540.0, ...}
   └── lunch   {'id': 2, 'bill': 320.0, ...}
```

### Приклад 2.3. Що таке `self`

**🔮 Прогноз:** чи однакові `Order.total(order)` і `order.total()`?

<details>
<summary>Відповідь</summary>

Так: `self` — це сам об'єкт; Python передає його першим аргументом. Так само, як `str.upper(district)` у прикладі 1.6.

</details>

In [ ]:
print(Order.total(order) == order.total())
print(Order.total(lunch), lunch.total())

Метод — звичайна функція з уроку 8, перший параметр якої — об'єкт. `order.total()` Python перетворює на `Order.total(order)`, тож усередині методу `self` — це `order`, а для `lunch.total()` — це `lunch`. Назва `self` — домовленість, а не ключове слово, але так пишуть усі.

### Приклад 2.4. Забули `self` у методі

Найчастіша помилка початківця — оголосити метод без параметра `self`.

In [ ]:
class Table:
    """Столик кафе."""

    def __init__(self, number, seats):
        self.number = number
        self.seats = seats

    def describe():                       # ← забули self
        return "столик"

    def fits(self, guests):
        return guests <= self.seats

In [ ]:
table = Table(5, seats=4)
print(table.fits(3), table.fits(6))

try:
    table.describe()
except TypeError as error:
    print("TypeError:", error)

`table.describe()` насправді — `Table.describe(table)`: Python передає **один** аргумент, а функція не приймає жодного. Звідси повідомлення «takes 0 positional arguments but 1 was given». Якщо бачиш таку помилку на методі без аргументів — шукай пропущений `self`.

`fits(self, guests)` написано правильно: `table.fits(3)` → `Table.fits(table, 3)`.

### Приклад 2.5. Атрибути можна змінювати — і помилитися в назві

Атрибути екземпляра — звичайні імена: їх можна перезаписати, і методи одразу побачать нове значення. А звернення до атрибута, якого немає, — `AttributeError`.

In [ ]:
print("до:", order.tip, order.total())
order.tip = 100.0
print("після:", order.tip, order.total())

try:
    print(order.tips)                     # одруківка: tips замість tip
except AttributeError as error:
    print("AttributeError:", error)

order.tip = 50.0                          # повертаємо, як було

- `total()` читає `self.tip` **під час виклику**, тож після `order.tip = 100.0` сума стала 640.0;
- одруківка `order.tips` не створює нічого «тихо» — Python одразу каже, що атрибута немає;
- зате **присвоєння** з одруківкою (`order.tips = 100`) спрацювало б мовчки й створило новий атрибут. Захищатися від цього вчимося в уроці 22 (інкапсуляція).

### Приклад 2.6. `__repr__`: як об'єкт себе показує

Без `__repr__` `print(order)` покаже адресу в пам'яті — для людини це марно:

In [ ]:
print(order)

Шістнадцяткове число — адреса об'єкта в пам'яті, і в тебе вона інша. Метод `__repr__` каже Python, як показувати об'єкт — у `print`, у списках, у налагоджувачі:

In [ ]:
class Order:
    """Замовлення кафе."""

    def __init__(self, order_id, time, bill, tip, guests):
        self.id = order_id
        self.time = time
        self.bill = bill
        self.tip = tip
        self.guests = guests

    def __repr__(self):
        return f"Order(№{self.id}, {self.time:%Y-%m-%d %H:%M}, {self.bill:.2f} грн)"

    def total(self):
        return self.bill + self.tip

In [ ]:
orders = [Order(1, datetime(2024, 7, 19, 18, 30), 540.0, 50.0, 2),
          Order(2, datetime(2024, 7, 19, 12, 10), 320.0, 30.0, 1)]
print(orders[0])
print(orders)

Тепер і окремий `print`, і список показують зрозумілі рядки. Імена з двома підкресленнями з обох боків — `__init__`, `__repr__` — **спеціальні методи** (dunder methods): Python викликає їх сам у певні моменти. Їм присвячено урок 24.

### Приклад 2.7. Метод викликає інший метод

Чек для гостей: сума з чайовими і скільки платить кожен. Новий метод не рахує все заново — він викликає вже готові методи через `self.`.

In [ ]:
class Order:
    """Замовлення кафе."""

    def __init__(self, order_id, time, bill, tip, guests):
        self.id = order_id
        self.time = time
        self.bill = bill
        self.tip = tip
        self.guests = guests

    def __repr__(self):
        return f"Order(№{self.id}, {self.time:%Y-%m-%d %H:%M}, {self.bill:.2f} грн)"

    def total(self):
        return self.bill + self.tip

    def per_guest(self):
        return round(self.bill / self.guests, 2)

    def share(self):
        """Скільки платить кожен гість — разом з чайовими."""
        return round(self.total() / self.guests, 2)

    def receipt(self):
        """Текст чека. Метод повертає рядок, а не друкує: друкує той, хто викликав."""
        return (f"Замовлення №{self.id} о {self.time:%H:%M}\n"
                f"  чек {self.bill:.2f} + чайові {self.tip:.2f} = {self.total():.2f} грн\n"
                f"  з кожного з {self.guests}: {self.share():.2f} грн")

In [ ]:
dinner = Order(3, datetime(2024, 7, 20, 20, 15), 980.0, 120.0, 4)
print(dinner.receipt())
print("per_guest:", dinner.per_guest(), "| share:", dinner.share())

`receipt()` викликає `self.total()` і `self.share()`, а `share()` — знову `self.total()`. Правило «сума = чек + чайові» записане **в одному місці**: зміниться воно (скажімо, додасться плата за обслуговування) — і `share()`, і `receipt()` порахують по-новому без жодних правок. `per_guest()` лишився, як був: 980 / 4 = 245.0 — частка чека без чайових.

### Перевір себе: картка водія

Заповни подумки картку для водія таксі: що він знає, що вміє, чого **не** робить?

<details>
<summary>Можлива відповідь</summary>

1. **Річ:** водій → `class Courier`; водій D-3 — екземпляр.
2. **Знає:** позивний `"D-3"`, скільки поїздок зробив.
3. **Вміє:** `deliver()` — поїздка, лічильник +1.
4. **Правило:** у кожного водія **свій** лічильник (пастку з цим побачимо в розділі 3).
5. **Не робить:** не вибирає замовлення й не рахує ціну за район — це `Delivery.fare()`.

</details>

### Приклад 2.8. Картка водія → код

Перекладаємо відповідь у клас. Лічильник поїздок створюємо в `__init__` — тоді він у кожного водія свій.

In [ ]:
class Courier:
    """Водій таксі «Смачно + Таксі»."""

    def __init__(self, name):
        self.name = name
        self.trips = 0                    # свій лічильник у кожного водія

    def __repr__(self):
        return f"Courier({self.name}, поїздок: {self.trips})"

    def deliver(self):
        self.trips += 1

In [ ]:
d3, d1 = Courier("D-3"), Courier("D-1")
d3.deliver()
d3.deliver()
d1.deliver()
print(d3, d1)

У D-3 дві поїздки, у D-1 — одна: лічильники незалежні, бо кожен `__init__` створює `trips` в **іншому** екземплярі. Чого клас **не** робить — теж видно з коду: у нього немає ні ціни, ні вибору замовлення.

### Задача 1. Клас `Customer`

Картка класу вже заповнена — переклади її в код (зразок — `Courier` вище):

| | `Customer` |
|---|---|
| **Що це?** | клієнт кафе |
| **Знає** | `name`, `phone`, `orders` — **власний** порожній список для кожного клієнта |
| **Вміє** | `add(order)` — додати замовлення; `spent()` — сума `bill` усіх замовлень; `__repr__` → `Customer(Оксана, замовлень: 2)` |
| **Правило** | у кожного клієнта свій список замовлень |
| **Не робить** | не створює замовлення й не видає їм номери |

Перевірка — у наступній клітинці: вона бере замовлення зі списку `orders` (приклад 2.6).

In [ ]:
class Customer:
    # BEGIN SOLUTION
    def __init__(self, name, phone):
        self.name = name
        self.phone = phone
        self.orders = []

    def __repr__(self):
        return f"Customer({self.name}, замовлень: {len(self.orders)})"

    def add(self, order):
        self.orders.append(order)

    def spent(self):
        return sum(order.bill for order in self.orders)
    # END SOLUTION

In [ ]:
oksana = Customer("Оксана", "050-123-4567")
taras = Customer("Тарас", "097-987-6543")
oksana.add(orders[0])
oksana.add(orders[1])
print(oksana, taras)
assert repr(oksana) == "Customer(Оксана, замовлень: 2)"
assert oksana.spent() == 860.0, "spent() — сума bill усіх замовлень клієнта"
assert taras.orders == [], "у кожного клієнта — свій список"
print("✅ Задача 1 розв'язана")

---
## 3. Атрибути класу й простір імен

Атрибут можна покласти не лише в екземпляр, а й у **сам клас**. Такий атрибут один на всіх — наприклад, вартість доставки за районами, однакова для всіх доставок.

### Приклад 3.1. Тариф — атрибут класу

In [ ]:
class Delivery:
    FEES = {"Поділ": 60, "Оболонь": 80, "Печерськ": 90}

    def __init__(self, order_id, district, driver):
        self.order_id = order_id
        self.district = district
        self.driver = driver

    def fare(self):
        return self.FEES[self.district]

In [ ]:
first = Delivery(1, "Оболонь", "D-3")
second = Delivery(2, "Поділ", "D-1")
print(first.fare(), second.fare())
print(first.FEES is second.FEES is Delivery.FEES)

`FEES` оголошено в тілі класу, поза методами, — це **атрибут класу**. `first.FEES`, `second.FEES` і `Delivery.FEES` — той самий словник (`is` → `True`): він існує в одному екземплярі на весь клас.

### Приклад 3.2. Де що лежить: `__dict__`

У кожного об'єкта свій **простір імен** — словник атрибутів `__dict__`. У класу теж.

In [ ]:
print(first.__dict__)
print("FEES" in Delivery.__dict__, "FEES" in first.__dict__)
print("fare" in Delivery.__dict__, "fare" in first.__dict__)

У `first` немає ні `FEES`, ні `fare`, але `first.FEES` і `first.fare()` працюють. Бо атрибут через крапку Python шукає **спершу в екземплярі, потім у класі**:

```text
first.FEES
   │
   ├─ є в first.__dict__?     ── так ──► беремо з екземпляра
   │   ні
   ├─ є в Delivery.__dict__?  ── так ──► беремо з класу
   │   ні
   └─ AttributeError
```

### Приклад 3.3. Атрибут екземпляра затінює атрибут класу

Акція: доставка №2 безкоштовна. Покладемо **цьому** екземпляру власний `FEES`.

**🔮 Прогноз:** що покаже `first.fare()` після `second.FEES = {"Поділ": 0}`? А `second.fare()` після `del second.FEES`?

<details>
<summary>Відповідь</summary>

`first.fare()` — як і було, 80: власний словник з'явився лише в `second`. Після `del second.FEES` пошук знову доходить до класу — і `second.fare()` повертає 60.

</details>

In [ ]:
second.FEES = {"Поділ": 0}
print("акція:      ", first.fare(), second.fare(), "| FEES у second:", "FEES" in second.__dict__)

del second.FEES
print("після акції:", first.fare(), second.fare(), "| FEES у second:", "FEES" in second.__dict__)

Присвоєння `second.FEES = …` **завжди** пише в екземпляр, а не в клас. Атрибут екземпляра «затуляє» однойменний атрибут класу — **затінює** його. Після `del` затінення зникає.

### Приклад 3.4. Змінити атрибут класу — бачать усі

Тариф подорожчав для всіх. Присвоюємо **класу** — і всі екземпляри без власного `FEES` одразу бачать нові ціни.

In [ ]:
old_fees = Delivery.FEES
Delivery.FEES = {"Поділ": 65, "Оболонь": 85, "Печерськ": 95}
print("новий тариф:", first.fare(), second.fare())

Delivery.FEES = old_fees                  # повертаємо, щоб решта уроку йшла з базовими цінами
print("як було:    ", first.fare(), second.fare())

Екземпляри не зберігають копію тарифу — вони щоразу шукають `FEES` у класі. Тому заміна атрибута класу діє на всіх одразу, навіть на вже створені доставки.

### Приклад 3.5. Пастка: присвоєння через `self`

Покладемо лічильник поїздок водія **в клас**, а не в `__init__`, як у прикладі 2.8.

**🔮 Прогноз:** що надрукує `print(courier.trips, Courier.trips)` після двох `deliver()`?

<details>
<summary>Відповідь</summary>

`2 0`: присвоєння `self.trips = …` створило атрибут в **екземплярі**, який затінив класовий. Щоб рахувати спільно — `Courier.trips += 1`.

</details>

In [ ]:
class Courier:
    trips = 0

    def __init__(self, name):
        self.name = name

    def deliver(self):
        self.trips += 1

In [ ]:
courier = Courier("D-3")
courier.deliver()
courier.deliver()
print(courier.trips, Courier.trips)
print(courier.__dict__)

`self.trips += 1` — це `self.trips = self.trips + 1`. **Читання** `self.trips` знайшло 0 у класі. А **присвоєння** `self.trips = …` записало 1 в екземпляр (приклад 3.3) — і далі цей атрибут затінює класовий. Для особистого лічильника водія результат правильний, але випадково: класове `trips` так і лишилося 0.

### Приклад 3.6. Спільний лічильник — правильно

Диспетчерська хоче знати і поїздки кожного водія, і скільки поїздок зробили **всі** разом. Особисте — в екземпляр, спільне — у клас, причому в клас пишемо **явно**.

In [ ]:
class Courier:
    total_trips = 0                       # спільний: один на всіх водіїв

    def __init__(self, name):
        self.name = name
        self.trips = 0                    # особистий: у кожного свій

    def deliver(self):
        self.trips += 1
        Courier.total_trips += 1          # явно в клас, а не через self

In [ ]:
d3, d1 = Courier("D-3"), Courier("D-1")
d3.deliver()
d3.deliver()
d1.deliver()
print(d3.trips, d1.trips, Courier.total_trips)
print(d3.__dict__)

У D-3 — 2, у D-1 — 1, разом — 3. У `d3.__dict__` немає `total_trips`: присвоєння `Courier.total_trips += 1` пише в клас, тож нічого не затінюється.

### Приклад 3.7. 🐛 Спільний кошик

Інша версія тієї самої пастки — **змінюваний** атрибут класу.

**🔮 Прогноз:** Оксана додала борщ у свій кошик. Що в кошику Тараса?

<details>
<summary>Відповідь</summary>

`['борщ']`. `self.items.append(...)` — не присвоєння, а зміна **того самого** списку, що лежить у класі. Затінення немає, і кошик у всіх один.

</details>

In [ ]:
class Cart:
    items = []                       # ← один список на весь клас

    def add(self, dish):
        self.items.append(dish)

In [ ]:
oksana_cart, taras_cart = Cart(), Cart()
oksana_cart.add("борщ")
print(taras_cart.items, taras_cart.items is Cart.items)

Виправлення: власні змінювані дані створюємо в `__init__` — тоді кожен виклик `Cart()` отримує новий порожній список.

In [ ]:
class Cart:
    def __init__(self):
        self.items = []              # новий список для кожного кошика

    def add(self, dish):
        self.items.append(dish)

In [ ]:
oksana_cart, taras_cart = Cart(), Cart()
oksana_cart.add("борщ")
print(oksana_cart.items, taras_cart.items)
assert taras_cart.items == [], "кошик Тараса має бути порожнім"

Правило: в атрибути класу кладемо лише те, що **справді спільне** й не змінюється через екземпляри (тарифи, константи). Списки, словники й лічильники кожного об'єкта — лише в `__init__`.

### Приклад 3.8. Клас — не оточення для методів

В уроці 19 внутрішня функція бачила змінні зовнішньої — правило LEGB, рівень E. Клас так **не** працює: метод не бачить атрибути класу просто за іменем.

In [ ]:
class Menu:
    VAT = 0.2

    def price_wrong(self, price):
        return price * (1 + VAT)          # шукає VAT за LEGB — і не знаходить

    def price_with_vat(self, price):
        return price * (1 + self.VAT)     # через крапку — знаходить у класі

In [ ]:
menu = Menu()
print(menu.price_with_vat(100), Menu.VAT)

try:
    menu.price_wrong(100)
except NameError as error:
    print("NameError:", error)

Метод шукає `VAT` за LEGB: локально, в оточуючих **функціях**, у модулі, серед вбудованих. Простору імен класу серед них немає. До атрибутів класу — лише через крапку: `self.VAT` або `Menu.VAT`.

---
## 4. Правила в `__init__`, `@classmethod` і `@staticmethod`

Четвертий рядок картки — **правила**: чек більший за 0, гостей хоча б один. Перевірки з уроку 14 тепер живуть у `__init__`: неправильне замовлення **неможливо створити**, хоч би звідки воно прийшло.

Замовлення часто приходять рядком каси (уроки 14–15). Створювати `Order` одразу з рядка допоможе **альтернативний конструктор** — `@classmethod`: метод отримує не екземпляр, а **сам клас** (`cls`) і повертає новий об'єкт. Правило, якому не потрібні ні екземпляр, ні клас («яка година — який прийом їжі»), оформлюють `@staticmethod`.

Нижче — повна версія `Order`, з якою працюватимемо до кінця уроку.

In [ ]:
class Order:
    """Замовлення кафе."""

    def __init__(self, order_id, time, bill, tip, guests):
        if bill <= 0:
            raise ValueError(f"сума чека має бути більшою за 0, а маємо {bill}")
        if guests < 1:
            raise ValueError(f"гостей має бути хоча б один, а маємо {guests}")
        self.id = order_id
        self.time = time
        self.bill = bill
        self.tip = tip
        self.guests = guests

    def __repr__(self):
        return f"Order(№{self.id}, {self.time:%Y-%m-%d %H:%M}, {self.bill:.2f} грн)"

    @classmethod
    def from_line(cls, line, order_id):
        """Рядок каси "2024-07-19 18:30;540.00;50;2" → Order."""
        fields = line.split(";")
        if len(fields) != 4:
            raise ValueError(f"очікували 4 поля, а маємо {len(fields)}")
        time_text, bill, tip, guests = fields
        return cls(order_id, datetime.strptime(time_text, "%Y-%m-%d %H:%M"),
                   float(bill), float(tip), int(guests))

    @staticmethod
    def meal_type(hour):
        if 11 <= hour <= 15:
            return "обід"
        if 17 <= hour <= 23:
            return "вечеря"
        return "інше"

    def meal(self):
        return self.meal_type(self.time.hour)

### Приклад 4.1. Неправильне замовлення не створиться

In [ ]:
attempts = [(540.0, 2), (0.0, 2), (610.0, 0)]
for bill, guests in attempts:
    try:
        created = Order(1, datetime(2024, 7, 21, 14, 20), bill, 50.0, guests)
        print("створено:", created)
    except ValueError as error:
        print("ValueError:", error)

Перше замовлення пройшло перевірки, друге й третє — ні: `raise` в `__init__` зупиняє створення, і об'єкта просто немає. Тепер кожен, хто має `Order`, може бути певен, що в ньому чек > 0 і гостей ≥ 1.

### Приклад 4.2. `@classmethod from_line` — замовлення з рядка каси

In [ ]:
order = Order.from_line("2024-07-21 21:30;1200.00;150;6", 7)
print(order, order.meal())
print(order.tip, order.guests, type(order.guests))

`Order.from_line(...)` викликаємо **на класі**: екземпляра ще немає, його лише треба створити. Python передає в `cls` сам клас `Order`, а `cls(...)` — це `Order(...)` з уже перетвореними типами (`float`, `int`, `datetime`). Перевірки `__init__` працюють і тут — бо `from_line` в кінці все одно викликає конструктор.

### Приклад 4.3. Поганий рядок каси

Дві різні помилки: неправильна кількість полів (перевіряє `from_line`) і нуль гостей (перевіряє `__init__`).

In [ ]:
for line in ["2024-07-21 14:20;610.00;60;0", "2024-07-21 14:20;610.00;60"]:
    try:
        Order.from_line(line, 8)
    except ValueError as error:
        print(f"{line!r} → {error}")

Кожна перевірка — там, де їй місце: формат рядка знає лише `from_line`, правила замовлення — `__init__`. Хоч би звідки прийшли дані — з каси, з CLI чи з тесту, — правила ті самі.

### Приклад 4.4. `@staticmethod` — правило без `self` і `cls`

`meal_type(hour)` не потребує ні даних замовлення, ні класу — лише годину. Але воно стосується теми класу, тому живе всередині нього. Викликати можна і на класі, і на екземплярі.

In [ ]:
for hour in (9, 12, 19):
    print(hour, "→", Order.meal_type(hour))

print(order.meal_type(13), order.meal())

`Order.meal_type(12)` — без жодного замовлення. `order.meal()` — звичайний метод: бере годину з `self.time` і передає її статичному методу.

| Вид методу | Перший параметр | Коли |
|---|---|---|
| звичайний | `self` — екземпляр | працює з даними конкретного об'єкта: `order.meal()` |
| `@classmethod` | `cls` — клас | створює об'єкти іншим способом: `Order.from_line(...)` |
| `@staticmethod` | немає | правило, що стосується теми класу, але не потребує його даних |

### Приклад 4.5. Чому `cls(...)`, а не `Order(...)`

`cls` — той клас, **через який** викликали метод. Це важливо, коли з'являються класи-нащадки (урок 21): замовлення з кіоску — теж замовлення.

In [ ]:
class KioskOrder(Order):
    """Замовлення з кіоску самообслуговування."""

In [ ]:
kiosk = KioskOrder.from_line("2024-07-22 12:05;210.00;0;1", 9)
print(type(kiosk).__name__, isinstance(kiosk, Order), kiosk.meal())

`from_line` написаний в `Order`, але викликали його через `KioskOrder`, тож `cls` — це `KioskOrder`, і повернувся саме `KioskOrder`. Якби всередині було `return Order(...)`, нащадок отримав би «чужий» тип.

---
## 5. `isinstance` і `type`

`type(obj)` повертає **точний** клас, `isinstance(obj, клас)` питає, чи є об'єкт екземпляром цього класу **або його нащадка**.

**🔮 Прогноз:** чи є `True` цілим числом?

<details>
<summary>Відповідь</summary>

`isinstance(True, int)` — `True`: `bool` — нащадок `int`. Але `type(True) is int` — `False`.

</details>

### Приклад 5.1. Точний клас чи «є екземпляром»

In [ ]:
print(type(order) is Order, isinstance(order, Order))
print(type(kiosk) is Order, isinstance(kiosk, Order))
print(isinstance(True, int), type(True) is int, bool.mro())

- `kiosk` — **не** точно `Order` (`type(...) is Order` → `False`), але є ним (`isinstance` → `True`), бо `KioskOrder` — нащадок;
- `True` — так само: `bool.mro()` — `bool → int → object`.

Для перевірок у коді зазвичай беруть `isinstance`: він не зламається, коли з'являться класи-нащадки.

### Приклад 5.2. Перевірка типу суми чека

`isinstance` приймає й **кортеж** класів: «чи це `int` або `float`». Але через приклад 5.1 `True` теж пройде таку перевірку — його треба відсіяти явно.

In [ ]:
def check_bill(value):
    """Сума чека має бути числом (int чи float), але не bool."""
    if isinstance(value, bool) or not isinstance(value, (int, float)):
        raise TypeError(f"сума чека має бути числом, а маємо {type(value).__name__}")
    return float(value)

In [ ]:
for value in [540, 540.5, "540", True]:
    try:
        print(repr(value), "→", check_bill(value))
    except TypeError as error:
        print(repr(value), "→ TypeError:", error)

`540` і `540.5` пройшли, рядок `"540"` — ні. Без окремої перевірки на `bool` значення `True` перетворилось би на чек 1.0 грн. `type(value).__name__` дає назву класу для зрозумілого повідомлення.

---
## 6. Архітектура: три способи тримати стан

Сервіс доставки можна організувати щонайменше трьома способами. Усі три правильні — різниться ціна змін і захищеність правил.

| | А. Функції над даними | Б. Замикання | В. Клас |
|---|---|---|---|
| Де стан | списки, що передають у функції | змінні фабрики | атрибути об'єкта |
| Хто стежить за правилами | кожен, хто викликає | функції фабрики | методи класу |
| Кілька незалежних сервісів | окремі списки | окремі виклики фабрики | окремі екземпляри |
| Додати нову дію | нова функція з тими самими параметрами | змінити фабрику | новий метод |
| Коли доречно | скрипт, разовий звіт | маленький стан, 1–2 дії | стан + багато дій + правила |

Спробуємо А і Б на одному правилі — «номери видає лише сервіс». Варіант В — розділ 7.

### Приклад 6.1. А. Функції над списком

In [ ]:
def add_order_a(orders, bill):
    """Номер — наступний за кількістю записів."""
    order_id = len(orders) + 1
    orders.append({"id": order_id, "bill": bill})
    return order_id

In [ ]:
book_a = []
add_order_a(book_a, 540.0)
add_order_a(book_a, 320.0)
book_a.append({"id": 1, "bill": -5.0})        # хтось дописав напряму, оминувши функцію
add_order_a(book_a, 980.0)
print([record["id"] for record in book_a])

Номери `[1, 2, 1, 4]`: дубль №1, пропущений №3 і від'ємний чек. Функція свої правила тримає, але список відкритий — кожен може оминути її одним `append`. Для разового звіту це не біда; для сервісу, який мають використовувати інші, — біда.

### Приклад 6.2. Б. Замикання

In [ ]:
def make_order_book():
    orders = {}
    next_id = 1

    def add(bill):
        nonlocal next_id
        if bill <= 0:
            raise ValueError(f"сума чека має бути більшою за 0, а маємо {bill}")
        orders[next_id] = bill
        next_id += 1
        return next_id - 1

    def count():
        return len(orders)

    return add, count

In [ ]:
add, count = make_order_book()
print(add(540.0), add(320.0))
try:
    add(-5.0)
except ValueError as error:
    print("ValueError:", error)
print("замовлень:", count())

Стан схований: до `orders` і `next_id` дістатися можна лише через `add` і `count`, тож правила не оминеш. Але кожна нова дія («звіт», «знайти замовлення») — це правка фабрики й ще одна функція в кортежі, а побачити стан для перевірки чи налагодження незручно. Клас поєднує обидві переваги: стан видно (`service.orders`), а змінюють його методи.

```text
шари системи (стрілки — лише вниз)

  інтерфейс   app.py: sys.argv, print, «Помилка: …»      (приклад 7.6)
      │
      ▼
  сервіс      DeliveryService: правила, звіти              (приклад 7.1)
      │
      ├──────────────► моделі: Order, Delivery — дані й перевірки
      └──────────────► сховище: data.json
```

Моделі нічого не знають ні про `print`, ні про `sys.argv`. Тому ті самі класи можна підключити до сайту чи Telegram-бота, не змінивши в них жодного рядка.

---
## 7. Розібраний приклад: `DeliveryService`

Спершу картка — кожен її рядок знайдеш у коді нижче:

| | `DeliveryService` |
|---|---|
| **Що це?** | сервіс однієї кав'ярні: тримає всі її замовлення й доставки |
| **Знає** | `orders` за номером, `deliveries` за номером замовлення, `next_id` |
| **Вміє** | `add_order(line)`, `add_delivery(order_id, district, driver)`, `report()` |
| **Правила** | номери видає лише сервіс; доставка лише до існуючого замовлення; одна доставка на замовлення |
| **Не робить** | не друкує й не читає `sys.argv` (інтерфейс), не рахує ціну за район (`Delivery`) |

Схема класів (текстом):

```text
DeliveryService                       Order                        Delivery
  orders: {id: Order}      1 ──── *     id, time, bill, tip,         FEES (атрибут класу)
  deliveries: {id: Delivery} 1 ── *     guests                       order_id, district, driver
  next_id: int                          from_line() (classmethod)    fare()
  add_order(), add_delivery(),          meal_type() (staticmethod)
  report()                              meal()                       order_id ┄┄► Order.id
```

### Приклад 7.1. Сервіс тримає стан і правила

In [ ]:
class DeliveryService:
    def __init__(self):
        self.orders = {}
        self.deliveries = {}
        self.next_id = 1

    def add_order(self, line):
        order = Order.from_line(line, self.next_id)
        self.orders[order.id] = order
        self.next_id += 1
        return order

    def add_delivery(self, order_id, district, driver):
        if order_id not in self.orders:
            raise ValueError(f"замовлення №{order_id} немає")
        if order_id in self.deliveries:
            raise ValueError(f"замовлення №{order_id} вже має доставку")
        delivery = Delivery(order_id, district, driver)
        self.deliveries[order_id] = delivery
        return delivery

    def report(self):
        cafe = sum(order.bill for order in self.orders.values())
        taxi = sum(delivery.fare() for delivery in self.deliveries.values())
        return (f"Замовлень: {len(self.orders)}, з доставкою: {len(self.deliveries)}; "
                f"кафе {cafe:.2f} грн, таксі {taxi} грн")

### Приклад 7.2. Сервіс у роботі

In [ ]:
smachno = DeliveryService()
smachno.add_order("2024-07-19 18:30;540.00;50;2")
smachno.add_order("2024-07-20 20:15;980.00;120;4")
smachno.add_delivery(1, "Оболонь", "D-3")

for attempt in [(1, "Поділ", "D-1"), (5, "Поділ", "D-1")]:
    try:
        smachno.add_delivery(*attempt)
    except ValueError as error:
        print("Помилка:", error)

print(smachno.orders)
print(smachno.report())

Що відбувається:

- `__init__` — стан сервісу: словники замовлень і доставок за номером (швидкий пошук, урок 17) і лічильник номерів;
- `add_order` — номер видає сам сервіс: `next_id` більше ніхто не рахує; метод **повертає** створене замовлення, а не друкує;
- `add_delivery` — правила «замовлення існує» і «одна доставка на замовлення» живуть у **єдиному** місці, через яке проходить кожна доставка. Друга доставка до №1 і доставка до неіснуючого №5 відхилені;
- `report` — `fare()` доставки бере ціну з атрибута класу `Delivery.FEES`: Оболонь — 80 грн.

### Приклад 7.3. Дві кав'ярні — два незалежні сервіси

In [ ]:
podil_cafe = DeliveryService()
podil_cafe.add_order("2024-07-19 12:00;250.00;20;1")

print(smachno.next_id, podil_cafe.next_id)
print(podil_cafe.report())

У `smachno` наступний номер — 3, у нової кав'ярні — 2: у кожного екземпляра свої `orders`, `deliveries` і `next_id`. У варіанті А (приклад 6.1) для цього довелося б передавати окремі списки в кожну функцію.

### Приклад 7.4. Невдалий рядок не «з'їдає» номер

**🔮 Прогноз:** у `smachno` наступний номер — 3. Що буде з `next_id`, якщо `add_order` отримає рядок з нулем гостей?

<details>
<summary>Відповідь</summary>

Лишиться 3. `Order.from_line` піднімає `ValueError` у першому ж рядку `add_order` — до `self.next_id += 1` виконання не доходить, тож номер не пропадає і замовлення в словник не потрапляє.

</details>

In [ ]:
try:
    smachno.add_order("2024-07-21 14:20;610.00;60;0")
except ValueError as error:
    print("Помилка:", error)

print("next_id:", smachno.next_id, "| замовлень:", len(smachno.orders))
third = smachno.add_order("2024-07-21 14:20;610.00;60;3")
print("наступне отримало номер", third.id)

Порядок рядків у методі — теж правило: **спершу перевірити й створити, потім змінювати стан**. Якби `next_id += 1` стояв першим, невдалий рядок лишив би «дірку» в нумерації.

### Приклад 7.5. Новий метод: `find(order_id)`

Сервіс має вміти знайти замовлення за номером. У файлі `.py` метод просто дописують у клас. У ноутбуці, щоб не переписувати клас цілком, функцію можна **приєднати** до вже оголошеного класу: `DeliveryService.find = find`. Перший параметр — так само `self`.

In [ ]:
def find(self, order_id):
    if order_id not in self.orders:
        raise ValueError(f"замовлення №{order_id} немає")
    return self.orders[order_id]


DeliveryService.find = find               # так роблять лише в ноутбуці; у файлі — метод у самому класі

In [ ]:
print(smachno.find(2), smachno.find(2).meal())
try:
    smachno.find(42)
except ValueError as error:
    print("Помилка:", error)

Метод з'явився в класі — отже, і в **усіх** його екземплярах, навіть створених раніше (`smachno`, `podil_cafe`): пошук іде в клас під час виклику (приклад 3.2).

### Приклад 7.6. Шар інтерфейсу: друкує лише він

Інтерфейс розбирає команду (тут — список, як `sys.argv[1:]`), викликає сервіс і друкує результат чи «Помилка: …». Правил у ньому немає — лише перетворення «команда ↔ виклик методу».

In [ ]:
def run_command(service, argv):
    """Інтерфейс: розбирає argv, викликає сервіс, друкує. Правил тут немає."""
    command, *args = argv
    try:
        if command == "order":
            print("Додано:", service.add_order(args[0]))
        elif command == "delivery":
            delivery = service.add_delivery(int(args[0]), args[1], args[2])
            print(f"Доставка до №{delivery.order_id}: {delivery.fare()} грн")
        elif command == "report":
            print(service.report())
        else:
            print("Невідома команда:", command)
    except ValueError as error:
        print("Помилка:", error)

In [ ]:
cafe = DeliveryService()
for argv in [["order", "2024-07-19 18:30;540.00;50;2"],
             ["delivery", "1", "Печерськ", "D-2"],
             ["delivery", "1", "Поділ", "D-1"],
             ["order", "зламаний рядок"],
             ["report"]]:
    run_command(cafe, argv)

Помилки, які підняли моделі й сервіс (`ValueError`), перетворюються на «Помилка: …» лише тут, на верхньому шарі. Захочемо Telegram-бота — напишемо інший `run_command`, а `Order`, `Delivery` і `DeliveryService` не зміняться.

### Задача 2. Звіт водіїв

Додай у `DeliveryService` метод `drivers()` — словник «водій → виторг» від більшого до меншого. Метод нічого не друкує; сервіс без доставок повертає `{}`. Приєднай його так само, як `find` у прикладі 7.5.

Підказка: `Counter` з уроку 13 уміє додавати: `totals[delivery.driver] += delivery.fare()`, а `most_common()` сортує від більшого.

In [ ]:
from collections import Counter


def drivers(self):
    # BEGIN SOLUTION
    totals = Counter()
    for delivery in self.deliveries.values():
        totals[delivery.driver] += delivery.fare()
    return dict(totals.most_common())
    # END SOLUTION


DeliveryService.drivers = drivers

In [ ]:
service = DeliveryService()
for line in ["2024-07-19 18:30;540.00;50;2", "2024-07-20 20:15;980.00;120;4", "2024-07-21 12:10;320.00;30;1"]:
    service.add_order(line)
service.add_delivery(1, "Оболонь", "D-3")
service.add_delivery(2, "Оболонь", "D-3")
service.add_delivery(3, "Поділ", "D-1")
print(service.drivers())
assert service.drivers() == {"D-3": 160, "D-1": 60}
assert list(service.drivers()) == ["D-3", "D-1"], "від більшого виторгу до меншого"
assert DeliveryService().drivers() == {}, "сервіс без доставок — порожній словник"
print("✅ Задача 2 розв'язана")

---
## 8. Промокод: від замикання до класу

### Приклад 8.1. Обмежений промокод — замикання з уроку 19

`LUCKY20` діє лише для перших трьох замовлень. Лічильник живе в замиканні:

In [ ]:
def make_limited_discount(percent, uses):
    left = uses

    def apply(bill):
        nonlocal left
        if left == 0:
            raise ValueError("промокод вичерпано")
        left -= 1
        return round(bill * (100 - percent) / 100, 2)

    return apply

In [ ]:
lucky = make_limited_discount(20, 3)
print(lucky(500.0), lucky(500.0), lucky(250.0))
try:
    lucky(500.0)
except ValueError as error:
    print("ValueError:", error)
print(lucky)

Працює, але: скільки використань лишилось — не видно (`print(lucky)` показує лише `<function ...>`), назви коду немає, а створити промокод з рядка «код:відсоток:використань» можна лише окремою функцією. Клас дасть атрибут `left`, `__repr__` і `@classmethod`.

### Задача 3. Промокод як клас

Спершу заповни подумки картку `Promo` (знає · вміє · правила · не робить), потім пиши код.

- `Promo(code, percent, uses)` — атрибути `code`, `percent`, `left`; `percent` поза 1–100 → `ValueError` уже в `__init__`;
- `apply(bill)` — повертає суму зі знижкою й зменшує `left`; коли використань не лишилось — `ValueError("промокод LUCKY20 вичерпано")` (з кодом промокоду);
- `__repr__` → `Promo(LUCKY20, 20%, лишилось 2)`;
- `@classmethod from_text("SUMMER10:10:100")` — промокод з рядка «код:відсоток:використань»;
- два промокоди мають незалежні лічильники.

In [ ]:
class Promo:
    # BEGIN SOLUTION
    def __init__(self, code, percent, uses):
        if not 1 <= percent <= 100:
            raise ValueError(f"знижка має бути від 1 до 100 %, а маємо {percent}")
        self.code = code
        self.percent = percent
        self.left = uses

    def __repr__(self):
        return f"Promo({self.code}, {self.percent}%, лишилось {self.left})"

    @classmethod
    def from_text(cls, text):
        code, percent, uses = text.split(":")
        return cls(code, int(percent), int(uses))

    def apply(self, bill):
        if self.left == 0:
            raise ValueError(f"промокод {self.code} вичерпано")
        self.left -= 1
        return round(bill * (100 - self.percent) / 100, 2)
    # END SOLUTION

In [ ]:
lucky = Promo("LUCKY20", percent=20, uses=3)
assert lucky.apply(500.0) == 400.0 and lucky.left == 2
assert repr(lucky) == "Promo(LUCKY20, 20%, лишилось 2)"
lucky.apply(500.0)
lucky.apply(250.0)
try:
    lucky.apply(500.0)
except ValueError as error:
    assert str(error) == "промокод LUCKY20 вичерпано"
else:
    raise AssertionError("очікували ValueError")
summer = Promo.from_text("SUMMER10:10:100")
assert (summer.code, summer.percent, summer.left) == ("SUMMER10", 10, 100)
assert lucky.left == 0 and summer.left == 100, "лічильники незалежні"
try:
    Promo("BAD", 150, 1)
except ValueError:
    pass
else:
    raise AssertionError("знижка 150% — ValueError")
print(summer)
print("✅ Задача 3 розв'язана")

<details>
<summary>Замикання чи клас — що зручніше перевіряти?</summary>

Клас довший, але стан видно (`lucky.left`), є зрозумілий `__repr__`, альтернативний конструктор і перевірка в одному місці. Тест може подивитися `left` напряму — у замиканні стан схований.

</details>

## ✅ Самоперевірка

1. Що відбувається при `Order(1, …)`?
2. Що таке `self`?
3. Чому `courier.trips` — 2, а `Courier.trips` — 0?
4. Чому метод не бачить `VAT` без `self.`?
5. Коли потрібен `@classmethod`, а коли `@staticmethod`?
6. Які 5 запитань ставиш до того, як написати `class`? Чому номер замовлення видає не `Order`?

<details>
<summary>Відповіді</summary>

1. Python створює об'єкт і викликає `__init__`, щоб записати атрибути (і перевірити дані).
2. Сам екземпляр, у якого викликали метод.
3. Присвоєння через `self` створило атрибут в екземплярі.
4. Простір імен класу не входить у LEGB методів.
5. `@classmethod` — альтернативний конструктор (`cls`); `@staticmethod` — правило без `self` і `cls`.
6. Що це за річ, що знає, що вміє, які правила тримає, чого не робить. Одне замовлення не бачить інших — вільний номер знає лише сервіс.

</details>

### Шпаргалка

```python
class Order:
    FEES = {...}                         # атрибут класу — один на всіх

    def __init__(self, order_id, bill):  # викликається при Order(...)
        self.id = order_id               # атрибути екземпляра
        self.items = []                  # змінювані дані — тільки тут!

    def __repr__(self):                  # як показувати об'єкт
        return f"Order(№{self.id})"

    def total(self):                     # метод: order.total() == Order.total(order)
        return ...

    @classmethod
    def from_line(cls, line): ...        # альтернативний конструктор

    @staticmethod
    def meal_type(hour): ...             # правило без self і cls

obj.__dict__                             # простір імен екземпляра
isinstance(obj, Order)                   # краще за type(obj) is Order
```

## Далі

- Практикум на реальних даних — [`lab_lesson_20_titanic_oop.ipynb`](https://github.com/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_2/lessons/lesson_20_classes_namespace/lab_lesson_20_titanic_oop.ipynb): клас `Passenger` для пасажирів «Титаніка».
- **Урок 21 — Наслідування, поліморфізм**: доставка таксі, кур'єром і самовивіз — різні класи з однаковим методом `fare()`.